# 🥇 Gold Layer - Sales Analytics
## AWS S3 External Storage via Unity Catalog

**Purpose**: Create business-ready sales analytics aggregates

**Source**: `workspace.`workspace-adventureworks-silver`` (S3)
**Target**: `workspace.`workspace-adventureworks-gold`` (S3)

**Gold Tables to Create**:
1. **gold_sales_by_product** - Sales metrics aggregated by product
2. **gold_sales_by_customer** - Sales metrics aggregated by customer
3. **gold_sales_by_date** - Daily/monthly/yearly sales trends
4. **gold_sales_kpi_summary** - High-level KPIs for dashboards

**Business Metrics**:
- Total revenue, quantity, order count
- Average order value, average unit price
- Year-over-year growth
- Customer acquisition and retention metrics

**Key Changes from Previous Version**:
- Uses new unified `fact_sales` table (combines order header and detail)
- Works with DATE columns directly instead of date surrogate keys
- Uses business IDs (customer_id, product_id) instead of surrogate keys
- Aggregates order-level totals before customer/date grouping

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql import Row
from datetime import datetime
import time
import pyspark.sql.functions as F

# Configuration
BRONZE_SCHEMA = "`workspace-adventureworks-bronze`"
SILVER_SCHEMA = "`workspace-adventureworks-silver`"
GOLD_SCHEMA = "`workspace-adventureworks-gold`"

print("=" * 80)
print("🥇 GOLD LAYER - SALES ANALYTICS")
print("=" * 80)
print(f"Start Time: {datetime.now()}")
print(f"Source: {SILVER_SCHEMA} (AWS S3)")
print(f"Target: {GOLD_SCHEMA} (AWS S3)")
print("=" * 80)
print()

transformation_results = []

In [0]:
# Sales Analytics by Product
print("\n" + "=" * 80)
print("📦 BUILDING gold_sales_by_product")
print("=" * 80)

start_time = time.time()

try:
    # Read from Silver
    fact_sales = spark.table(f"{SILVER_SCHEMA}.fact_sales")
    dim_product_current = spark.table(f"{SILVER_SCHEMA}.dim_product").filter("is_current = true")
    
    # Join tables
    joined_data = fact_sales.join(dim_product_current, "product_id", "inner")
    
    # Add sales year
    sales_with_year = joined_data.withColumn("sales_year", F.year(col("order_date")))
    
    # Aggregate by product
    gold_sales_by_product = (sales_with_year
        .groupBy(
            "product_id",
            "product_name",
            "product_number",
            "category_name",
            "color",
            "list_price",
            "sales_year"
        )
        .agg(
            F.sum("order_quantity").alias("total_quantity_sold"),
            F.count("sales_order_id").alias("total_orders"),
            F.sum("line_total").alias("total_revenue"),
            F.avg("unit_price").alias("avg_unit_price"),
            F.min("unit_price").alias("min_unit_price"),
            F.max("unit_price").alias("max_unit_price"),
            F.countDistinct("customer_id").alias("unique_customers")
        )
        .withColumn("avg_order_quantity", col("total_quantity_sold") / col("total_orders"))
        .withColumn("revenue_per_unit", col("total_revenue") / col("total_quantity_sold"))
    )
    
    # Write to Gold
    gold_sales_by_product.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD_SCHEMA}.gold_sales_by_product")
    
    duration = time.time() - start_time
    row_count = gold_sales_by_product.count()
    
    transformation_results.append({
        "table": "gold_sales_by_product",
        "status": "success",
        "row_count": row_count,
        "duration_seconds": round(duration, 2)
    })
    
    print(f"\n✅ gold_sales_by_product created: {row_count:,} rows in {duration:.2f}s")
    
except Exception as e:
    duration_on_error = time.time() - start_time
    error_msg = str(e)
    print(f"\n❌ Failed: {error_msg}")
    transformation_results.append({"table": "gold_sales_by_product", "status": "failed", "row_count": 0, "duration_seconds": round(duration_on_error, 2)})

In [0]:
# Sales Analytics by Customer
print("\n" + "=" * 80)
print("👥 BUILDING gold_sales_by_customer")
print("=" * 80)

start_time = time.time()

try:
    # Read from Silver
    fact_sales = spark.table(f"{SILVER_SCHEMA}.fact_sales")
    dim_customer = spark.table(f"{SILVER_SCHEMA}.dim_customer").filter(col("is_current") == True)
    
    # Aggregate sales by order to get order-level totals
    order_totals = (fact_sales
        .groupBy("sales_order_id", "customer_id", "order_date")
        .agg(
            F.sum("line_total").alias("order_line_total"),
            F.first("order_subtotal").alias("order_subtotal"),
            F.first("tax_amount").alias("order_tax"),
            F.first("freight").alias("order_freight"),
            F.first("total_due").alias("order_total_due")
        )
    )
    
    # Join with customer dimension to get account number
    joined_data = (order_totals
        .join(
            dim_customer.select("customer_id", "account_number"),
            "customer_id",
            "inner"
        )
    )
    
    # Aggregate by customer
    gold_sales_by_customer = (joined_data
        .withColumn("sales_year", F.year(col("order_date")))
        .groupBy(
            "customer_id",
            "account_number",
            "sales_year"
        )
        .agg(
            F.count("sales_order_id").alias("total_orders"),
            F.sum("order_total_due").alias("total_revenue"),
            F.avg("order_total_due").alias("avg_order_value"),
            F.sum("order_subtotal").alias("total_subtotal"),
            F.sum("order_tax").alias("total_tax"),
            F.sum("order_freight").alias("total_freight"),
            F.min("order_date").alias("first_order_date"),
            F.max("order_date").alias("last_order_date")
        )
        .withColumn("avg_subtotal_per_order", col("total_subtotal") / col("total_orders"))
    )
    
    # Write to Gold
    (gold_sales_by_customer
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD_SCHEMA}.gold_sales_by_customer")
    )
    
    duration = time.time() - start_time
    row_count = gold_sales_by_customer.count()
    
    transformation_results.append({
        "table": "gold_sales_by_customer",
        "status": "success",
        "row_count": row_count,
        "duration_seconds": round(duration, 2)
    })
    
    print(f"\n✅ gold_sales_by_customer created: {row_count:,} rows in {duration:.2f}s")
    
except Exception as e:
    print(f"\n❌ Failed: {str(e)}")
    transformation_results.append({"table": "gold_sales_by_customer", "status": "failed", "row_count": 0, "duration_seconds": round(time.time() - start_time, 2)})

In [0]:
# Sales Analytics by Date (Time Series)
print("\n" + "=" * 80)
print("📅 BUILDING gold_sales_by_date")
print("=" * 80)

start_time = time.time()

try:
    # Read from Silver
    fact_sales = spark.table(f"{SILVER_SCHEMA}.fact_sales")
    dim_date = spark.table(f"{SILVER_SCHEMA}.dim_date")
    
    # Aggregate sales by order to get order-level totals
    order_totals = (fact_sales
        .groupBy("sales_order_id", "order_date", "customer_id", "is_online_order")
        .agg(
            sum("line_total").alias("order_line_total"),
            first("total_due").alias("order_total_due")
        )
    )
    
    # Aggregate by date
    gold_sales_by_date = (order_totals
        .join(dim_date, order_totals.order_date == dim_date.date_value)
        .groupBy(
            col("date_sk"),
            col("date_value"),
            col("year"),
            col("quarter"),
            col("month"),
            col("month_name"),
            col("week_of_year"),
            col("day_name"),
            col("is_weekend")
        )
        .agg(
            count("sales_order_id").alias("order_count"),
            sum("order_total_due").alias("total_revenue"),
            avg("order_total_due").alias("avg_order_value"),
            countDistinct("customer_id").alias("unique_customers"),
            sum(when(col("is_online_order") == True, 1).otherwise(0)).alias("online_order_count"),
            sum(when(col("is_online_order") == False, 1).otherwise(0)).alias("offline_order_count")
        )
        .withColumn("online_order_pct", col("online_order_count") / col("order_count") * 100)
    )
    
    # Add year-over-year comparison
    window_yoy = Window.partitionBy(col("month"), col("day_of_month")).orderBy("year")
    
    gold_sales_by_date = (gold_sales_by_date
        .join(
            dim_date.select("date_sk", col("day_of_month")),
            "date_sk"
        )
        .withColumn("prev_year_revenue", lag("total_revenue", 1).over(window_yoy))
        .withColumn("yoy_revenue_growth", 
            when(col("prev_year_revenue").isNotNull(), 
                (col("total_revenue") - col("prev_year_revenue")) / col("prev_year_revenue") * 100
            ).otherwise(None)
        )
        .drop("day_of_month")
    )
    
    # Write to Gold
    (gold_sales_by_date
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD_SCHEMA}.gold_sales_by_date")
    )
    
    duration = time.time() - start_time
    row_count = gold_sales_by_date.count()
    
    transformation_results.append({
        "table": "gold_sales_by_date",
        "status": "success",
        "row_count": row_count,
        "duration_seconds": round(duration, 2)
    })
    
    print(f"\n✅ gold_sales_by_date created: {row_count:,} rows in {duration:.2f}s")
    
except Exception as e:
    print(f"\n❌ Failed: {str(e)}")
    transformation_results.append({"table": "gold_sales_by_date", "status": "failed", "row_count": 0, "duration_seconds": round(time.time() - start_time, 2)})

In [0]:
# Sales KPI Summary (High-Level Metrics)
print("\n" + "=" * 80)
print("📊 BUILDING gold_sales_kpi_summary")
print("=" * 80)

start_time = time.time()

try:
    # Read from Silver
    fact_sales = spark.table(f"{SILVER_SCHEMA}.fact_sales")
    
    # Aggregate sales by order to get order-level totals
    order_totals = (fact_sales
        .groupBy("sales_order_id", "order_date", "customer_id")
        .agg(
            sum("line_total").alias("order_line_total"),
            first("order_subtotal").alias("order_subtotal"),
            first("tax_amount").alias("order_tax"),
            first("freight").alias("order_freight"),
            first("total_due").alias("order_total_due")
        )
        .withColumn("year", year("order_date"))
        .withColumn("month", month("order_date"))
    )
    
    # Calculate KPIs by year/month
    gold_sales_kpi = (order_totals
        .groupBy(col("year"), col("month"))
        .agg(
            count("sales_order_id").alias("total_orders"),
            sum("order_total_due").alias("total_revenue"),
            sum("order_subtotal").alias("total_subtotal"),
            sum("order_tax").alias("total_tax"),
            sum("order_freight").alias("total_freight"),
            avg("order_total_due").alias("avg_order_value"),
            countDistinct("customer_id").alias("unique_customers")
        )
        .withColumn("revenue_per_customer", col("total_revenue") / col("unique_customers"))
        .withColumn("orders_per_customer", col("total_orders") / col("unique_customers"))
    )
    
    # Add month-over-month growth
    window_mom = Window.orderBy("year", "month")
    
    gold_sales_kpi = (gold_sales_kpi
        .withColumn("prev_month_revenue", lag("total_revenue", 1).over(window_mom))
        .withColumn("mom_revenue_growth_pct",
            when(col("prev_month_revenue").isNotNull(),
                (col("total_revenue") - col("prev_month_revenue")) / col("prev_month_revenue") * 100
            ).otherwise(None)
        )
    )
    
    # Write to Gold
    (gold_sales_kpi
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD_SCHEMA}.gold_sales_kpi_summary")
    )
    
    duration = time.time() - start_time
    row_count = gold_sales_kpi.count()
    
    transformation_results.append({
        "table": "gold_sales_kpi_summary",
        "status": "success",
        "row_count": row_count,
        "duration_seconds": round(duration, 2)
    })
    
    print(f"\n✅ gold_sales_kpi_summary created: {row_count:,} rows in {duration:.2f}s")
    
except Exception as e:
    print(f"\n❌ Failed: {str(e)}")
    transformation_results.append({"table": "gold_sales_kpi_summary", "status": "failed", "row_count": 0, "duration_seconds": round(time.time() - start_time, 2)})

In [0]:
# Summary
print("\n" + "=" * 80)
print("📊 SALES ANALYTICS SUMMARY")
print("=" * 80)

success_count = sum(1 for r in transformation_results if r["status"] == "success")
failed_count = sum(1 for r in transformation_results if r["status"] == "failed")
total_rows = sum(r["row_count"] for r in transformation_results)

print(f"\nGold Tables Created: {len(transformation_results)}")
print(f"  ✅ Success: {success_count}")
print(f"  ❌ Failed: {failed_count}")
print(f"Total Rows Created: {total_rows:,}")

summary_df = spark.createDataFrame([Row(**r) for r in transformation_results])
print("\nDetailed Results:")
display(summary_df.orderBy("status", "table"))

print("\n" + "=" * 80)
if failed_count == 0:
    print("✅ GOLD SALES ANALYTICS COMPLETED SUCCESSFULLY")
    print("=" * 80)
    print(f"Completion Time: {datetime.now()}")
    dbutils.notebook.exit("SUCCESS")
else:
    print("⚠️ GOLD SALES ANALYTICS COMPLETED WITH ERRORS")
    print("=" * 80)
    print(f"Completion Time: {datetime.now()}")
    dbutils.notebook.exit(f"PARTIAL_SUCCESS: {failed_count} tables failed")